# 4. External validation on Reactome and MeSH

1. Build the two sets
2. Embed them with every model (GPU).
3. Evaluate with the pooling chosen on the KEGG validation split.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import glob, json, os

REPO = 'https://github.com/errory230/pathway-supervised-embedder.git'
CODE = '/content/pathway-supervised-embedder'
BASE = '/content/drive/MyDrive/kegg-bge'
DATA = '/content/data/splits'

if os.path.isdir(CODE):
    !git -C {CODE} pull -q
else:
    !git clone -q {REPO} {CODE}
S = f'{CODE}/scripts'
!mkdir -p /content/data {BASE}
!cp -r {BASE}/data/splits /content/data/
def latest(d): return sorted(glob.glob(f'{d}/run-*'))[-1]

EXT = f'{BASE}/external'
EMAIL = 'your@email.com' # <=== EMAIL Here
API_KEY = ''
key = f'--api-key {API_KEY}' if API_KEY else ''

## 1. Build the sets


In [ ]:
!python {S}/build_mesh.py --email {EMAIL} {key} --kegg-dir {DATA} --out-dir {EXT}/mesh

In [ ]:
!python {S}/build_reactome.py --email {EMAIL} {key} --kegg-dir {DATA} --out-dir {EXT}/reactome

## 2. Embeddings (GPU)

In [ ]:
runs = json.load(open(f'{BASE}/bienc_runs.json'))
custom = ' '.join(f'{k}={runs[k]}/best' for k in ['kegg_bge_s42', 'kegg_bge_s1', 'kegg_bge_s2'])
CROSS = sorted(glob.glob(f'{BASE}/result_crossencoder/run-*'))
ours = f'--ours {CROSS[-1]}/best' if CROSS else ''
for db in ['mesh', 'reactome']:
    !python {S}/embed.py --data-dir {EXT}/{db} --train-dir {DATA} --splits test --out-dir {BASE}/emb_{db} --bf16 {ours} --custom {custom} \
        --models longformer biobert scibert pubmedbert specter medcpt bge tfidf random
    !python {S}/embed.py --data-dir {EXT}/{db} --train-dir {DATA} --splits test --out-dir {BASE}/emb_{db} --bf16 --input native --models specter medcpt

## 3. Evaluation


In [ ]:
POOL = f'{BASE}/eval/pooling_choice.json'
!pip -q install tabulate
!python {S}/evaluate.py --data-dir {EXT}/mesh --emb-dir {BASE}/emb_mesh --out-dir {BASE}/eval_mesh \
    --level-keys level1,level2,level3,level4 --pooling-from {POOL} --reference kegg_bge_s42
!python {S}/evaluate.py --data-dir {EXT}/reactome --emb-dir {BASE}/emb_reactome --out-dir {BASE}/eval_reactome \
    --pooling-from {POOL} --reference kegg_bge_s42
from IPython.display import Markdown, display
for db in ['mesh', 'reactome']:
    print(json.dumps(json.load(open(f'{EXT}/{db}/meta.json')), indent=1, ensure_ascii=False)[:1500])
    display(Markdown(open(f'{BASE}/eval_{db}/report_test.md').read()))